# Emotion Recognition System — Dataset Quality Analysis

**Internship:** NIT Sikkim  
This notebook audits the provided YOLO dataset before preprocessing or model training. It treats `train`, `valid`, and `test` as separate splits.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import math

import cv2
import numpy as np
import pandas as pd
import yaml
from PIL import Image

pd.set_option('display.max_colwidth', None)

In [ ]:
dataset_path = Path('../dataset/YOLO_format').resolve()
if not dataset_path.exists():
    dataset_path = Path('dataset/YOLO_format').resolve()

with (dataset_path / 'data.yaml').open(encoding='utf-8') as file:
    data_config = yaml.safe_load(file)

class_names = data_config['names']
if isinstance(class_names, dict):
    class_names = [class_names[index] for index in sorted(class_names)]

splits = {
    name: {'images': dataset_path / name / 'images', 'labels': dataset_path / name / 'labels'}
    for name in ('train', 'valid', 'test')
}

print('Dataset:', dataset_path)
print('Classes:', dict(enumerate(class_names)))
for name, folders in splits.items():
    print(f"{name}: images={folders['images'].exists()}, labels={folders['labels'].exists()}")

## 1. Dataset structure

The configured folders and emotion-class mapping are displayed above. The following commits add image, label, duplication, and sharpness quality checks.

## 2. Image properties and resolution

Every image is decoded to check readability while recording dimensions, format, colour mode, channels, and file size.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def image_files(folder):
    return sorted(path for path in folder.iterdir()
                  if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)

def analyze_images(paths):
    result = {'total_images': len(paths), 'readable_images': 0, 'corrupt_images': 0,
              'sizes': Counter(), 'formats': Counter(), 'modes': Counter(),
              'channels': Counter(), 'file_sizes_bytes': [], 'records': [], 'errors': []}
    for path in paths:
        try:
            with Image.open(path) as image:
                image.load()
                width, height = image.size
                mode = image.mode
                channels = len(image.getbands())
                result['readable_images'] += 1
                result['sizes'][(width, height)] += 1
                result['formats'][image.format or 'Unknown'] += 1
                result['modes'][mode] += 1
                result['channels'][channels] += 1
                result['file_sizes_bytes'].append(path.stat().st_size)
                result['records'].append({'path': path, 'width': width, 'height': height,
                                          'mode': mode, 'channels': channels,
                                          'format': image.format or 'Unknown',
                                          'file_size_bytes': path.stat().st_size})
        except Exception as error:
            result['corrupt_images'] += 1
            result['errors'].append((path.name, str(error)))
    return result

split_images = {name: image_files(folders['images']) for name, folders in splits.items()}
image_analysis = {name: analyze_images(paths) for name, paths in split_images.items()}

for name, result in image_analysis.items():
    print(f'\n{name.upper()}')
    print('Total/readable/corrupt:', result['total_images'], result['readable_images'], result['corrupt_images'])
    print('Resolutions:', dict(result['sizes']))
    print('Formats:', dict(result['formats']))
    print('Modes:', dict(result['modes']))
    print('Channels:', dict(result['channels']))

## 3. Image–label pairs and YOLO annotations

Every image should have a same-stem `.txt` file. Valid YOLO rows contain `class_id x_center y_center width height`, with finite normalized coordinates and positive width/height.

In [ ]:
def check_pairs(image_paths, label_folder):
    image_by_stem = {path.stem: path for path in image_paths}
    label_paths = sorted(path for path in label_folder.glob('*.txt') if path.is_file())
    label_by_stem = {path.stem: path for path in label_paths}
    missing = sorted(set(image_by_stem) - set(label_by_stem))
    orphan = sorted(set(label_by_stem) - set(image_by_stem))
    return image_by_stem, label_by_stem, missing, orphan

def analyze_labels(label_paths, class_count):
    class_counts = Counter()
    invalid_rows, empty_files, annotations = [], [], 0
    for path in label_paths.values():
        nonempty_lines = [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
        if not nonempty_lines:
            empty_files.append(path.name)
        for line_number, line in enumerate(nonempty_lines, start=1):
            fields = line.split()
            try:
                values = [float(value) for value in fields]
                class_id = int(values[0])
                valid = (len(values) == 5 and values[0].is_integer() and 0 <= class_id < class_count
                         and all(math.isfinite(value) for value in values)
                         and all(0 <= value <= 1 for value in values[1:])
                         and values[3] > 0 and values[4] > 0)
                if not valid:
                    raise ValueError('not a valid normalized YOLO row')
                class_counts[class_id] += 1
                annotations += 1
            except (ValueError, IndexError):
                invalid_rows.append((path.name, line_number, line))
    return {'annotations': annotations, 'class_counts': class_counts,
            'invalid_rows': invalid_rows, 'empty_files': empty_files}

pair_analysis, label_analysis = {}, {}
for name, folders in splits.items():
    images, labels, missing, orphan = check_pairs(split_images[name], folders['labels'])
    pair_analysis[name] = {'images': images, 'labels': labels, 'missing_labels': missing, 'orphan_labels': orphan}
    label_analysis[name] = analyze_labels(labels, len(class_names))
    print(f"\n{name.upper()}: missing labels={len(missing)}, orphan labels={len(orphan)}, empty labels={len(label_analysis[name]['empty_files'])}, invalid rows={len(label_analysis[name]['invalid_rows'])}")
    print('Annotations by emotion:', {class_names[key]: value for key, value in sorted(label_analysis[name]['class_counts'].items())})

## 4. Exact duplicate detection

MD5 hashes find byte-for-byte duplicate files. This checks duplicates inside each split and exact leakage between each pair of splits; it does not claim to find visually similar recompressed images.

In [ ]:
def file_hash(path, chunk_size=1024 * 1024):
    digest = hashlib.md5()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

hash_maps = {}
for name, paths in split_images.items():
    hashes = defaultdict(list)
    for path in paths:
        hashes[file_hash(path)].append(path)
    hash_maps[name] = hashes
    within_split = [group for group in hashes.values() if len(group) > 1]
    print(f'{name.upper()} exact duplicates within split:', sum(len(group) - 1 for group in within_split))

cross_split_duplicates = {}
split_names = list(splits)
for index, left in enumerate(split_names):
    for right in split_names[index + 1:]:
        shared_hashes = sorted(set(hash_maps[left]) & set(hash_maps[right]))
        matches = [(hash_maps[left][digest], hash_maps[right][digest]) for digest in shared_hashes]
        cross_split_duplicates[(left, right)] = matches
        print(f'Exact duplicates {left} ↔ {right}:', len(matches))